# 10. Scaled Agent Memory: Vector Memory vs. a Bi-Temporal Graph Memory on LongMemEval

**Presentation map:** Slide 52 (memory tiers: episodic → semantic), Slide 53 (bi-temporal edges — a contradiction
*invalidates* the old edge instead of deleting it), Slide 54 (Graphiti-style `add_episode`: extract → resolve →
invalidate), Slide 55 (Zep/Graphiti on LongMemEval: 71.2% vs 60.2% with full context, +38.4% on temporal reasoning,
but single-session-assistant fell from 94.6% to 80.4%), Slide 56 (no memory design wins every question type),
Slide 38 (keep the source text attached).

This is the scaled companion to Notebook 4: instead of hand-written runbook questions, it uses the public
**LongMemEval** benchmark (Wu et al., ICLR 2025).

**Dataset:** `data/benchmarks/longmemeval/longmemeval_s_cleaned.json` — the *cleaned* LongMemEval-S release
(Hugging Face `xiaowu0162/longmemeval-cleaned`, MIT licence). 500 questions; each comes with its own chat history of
≈48 user–assistant sessions (≈115k tokens), timestamped, in which 1–3 "evidence" sessions are hidden among filler
sessions. Question types: single-session-user / -assistant / -preference, multi-session, knowledge-update,
temporal-reasoning, plus 30 abstention questions (the answer is *not* in the history).

**What we compare** on a stratified sample of 108 questions, with the same answer model (`gpt-4o-mini`) and prompt:

| Memory | Index | What the answerer sees |
|---|---|---|
| (a) **Vector memory** | every conversation round embedded with its timestamp | top-10 rounds |
| (b) **Bi-temporal graph memory** (Graphiti-like) | LLM extracts entity–relation facts per session; entity resolution; state facts invalidate older ones (`invalid_at`), never deleted; each fact keeps its source session + turn | top-25 facts with validity dates + the source turns of the top-5 facts |
| (c) **Full context** (subset of 28) | none | the whole history (oldest sessions dropped if > 118k tokens) |

**Scoring** follows LongMemEval's official evaluator: the same type-specific yes/no judge prompts, copied from
`src/evaluation/evaluate_qa.py`. *Deviation:* the judge is `gpt-4o-mini`, not `gpt-4o`.

> All LLM outputs are cached in `data/benchmarks/longmemeval/cache/`. With `RUN_LIVE = False` (default) the notebook
> replays the saved run in seconds; `RUN_LIVE = True` recomputes anything missing (≈20M input tokens, ≈30 min).

In [1]:
import json, os, re, time, random, hashlib, math
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, Literal
import numpy as np
from pydantic import BaseModel, Field
from openai import OpenAI

RUN_LIVE = False               # True: recompute anything missing from the cache (costs API tokens)
DATA = "data/benchmarks/longmemeval"
CACHE = f"{DATA}/cache"
os.makedirs(CACHE, exist_ok=True)

CHAT_MODEL, EMBED_MODEL = "gpt-4o-mini", "text-embedding-3-small"
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "missing"), max_retries=2, timeout=180)

def cache_load(name):
    path = f"{CACHE}/{name}.jsonl"
    if not os.path.exists(path):
        return {}
    rows = [json.loads(l) for l in open(path)]
    return {r["key"]: r for r in rows}

def cache_append(name, row):
    with open(f"{CACHE}/{name}.jsonl", "a") as f:
        f.write(json.dumps(row) + "\n")

def need_live(what):
    if not RUN_LIVE:
        raise RuntimeError(f"{what} is not in the cache - set RUN_LIVE = True to compute it (uses the OpenAI API).")

def with_retry(fn, *a, **kw):
    for attempt in range(8):
        try:
            return fn(*a, **kw)
        except Exception as e:                      # rate limits, timeouts, transient 5xx
            if attempt == 7:
                raise
            time.sleep(min(60, 2 ** attempt + random.random()))

def pmap(fn, items, workers=16):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(fn, items))

def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

print("RUN_LIVE =", RUN_LIVE, "| cache files:", sorted(os.listdir(CACHE)))

RUN_LIVE = False | cache files: ['answers.jsonl', 'extractions.jsonl', 'judgments.jsonl', 'retrieval_graph.jsonl', 'retrieval_vector.jsonl']


## The benchmark and a stratified sample

Each LongMemEval question carries its own haystack, so every question is a separate memory problem. We sample with a
fixed seed: **16 questions from each of the 6 types** (abstention items excluded) plus **12 abstention questions**
(3 from each type that has them) — 108 questions, ≈5,100 sessions, ≈13M tokens of history.

In [2]:
data = json.load(open(f"{DATA}/longmemeval_s_cleaned.json"))
by_id = {q["question_id"]: q for q in data}
is_abs = lambda q: q["question_id"].endswith("_abs")
print(f"LongMemEval-S (cleaned): {len(data)} questions;",
      dict(Counter(q["question_type"] for q in data)), f"| abstention: {sum(map(is_abs, data))}")

rng = random.Random(7)
TYPES = ["single-session-user", "single-session-assistant", "single-session-preference",
         "multi-session", "knowledge-update", "temporal-reasoning"]
sample = []
for t in TYPES:
    pool = sorted(q["question_id"] for q in data if q["question_type"] == t and not is_abs(q))
    sample += rng.sample(pool, 16)
for t in ["single-session-user", "multi-session", "knowledge-update", "temporal-reasoning"]:
    pool = sorted(q["question_id"] for q in data if q["question_type"] == t and is_abs(q))
    sample += rng.sample(pool, 3)
group = lambda qid: "abstention" if is_abs(by_id[qid]) else by_id[qid]["question_type"]

n_sess = sum(len(by_id[q]["haystack_sessions"]) for q in sample)
n_chars = sum(len(t["content"]) for q in sample for s in by_id[q]["haystack_sessions"] for t in s)
print(f"Sample: {len(sample)} questions, {n_sess:,} sessions, ~{n_chars / 4 / 1e6:.1f}M tokens of history")
print("Per group:", dict(Counter(map(group, sample))))
ex = by_id[sample[64]]
print(f"\nExample ({ex['question_type']}): {ex['question']!r}  asked {ex['question_date']}  -> gold {ex['answer']!r}")

LongMemEval-S (cleaned): 500 questions; {'single-session-user': 70, 'multi-session': 133, 'single-session-preference': 30, 'temporal-reasoning': 133, 'knowledge-update': 78, 'single-session-assistant': 56} | abstention: 30
Sample: 108 questions, 5,107 sessions, ~13.2M tokens of history
Per group: {'single-session-user': 16, 'single-session-assistant': 16, 'single-session-preference': 16, 'multi-session': 16, 'knowledge-update': 16, 'temporal-reasoning': 16, 'abstention': 12}

Example (knowledge-update): 'How many women are on the team led by my former manager Rachel?'  asked 2023/08/03 (Thu) 06:02  -> gold '6'


## (a) Vector memory — embed every conversation round with its timestamp

A *round* is one user message plus the assistant reply. Each round is embedded with its session date in front, and
the 10 most similar rounds (each cut to 1,500 characters) are handed to the answerer. This is the strong, cheap
baseline of slides 32 and 68.

In [3]:
ROUND_CAP, K_VEC = 1500, 10

def rounds_of(q):
    out = []
    for sid, date, sess in zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"]):
        for i in range(0, len(sess), 2):
            txt = "\n".join(f"{t['role'].upper()}: {t['content']}" for t in sess[i:i + 2])
            out.append({"sid": sid, "date": date, "text": f"[{date}] {txt}"})
    return out

def embed(texts, cap=20000):
    vecs = []
    batch, size = [], 0
    def flush():
        r = with_retry(client.embeddings.create, model=EMBED_MODEL, input=batch)
        vecs.extend(e.embedding for e in r.data)
        return r.usage.total_tokens
    used = 0
    for t in texts:
        t = t[:cap] or " "
        if batch and (len(batch) >= 512 or size + len(t) > 600_000):
            used += flush(); batch, size = [], 0
        batch.append(t); size += len(t)
    if batch:
        used += flush()
    v = np.array(vecs, dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True), used

vec_cache = cache_load("retrieval_vector")
def vector_retrieve(qid):
    if qid in vec_cache:
        return vec_cache[qid]
    need_live("vector retrieval")
    q = by_id[qid]
    rs = rounds_of(q)
    E, used = embed([r["text"] for r in rs] + [q["question"]])
    sims = E[:-1] @ E[-1]
    top = np.argsort(-sims)[:K_VEC]
    top = sorted(top, key=lambda i: i)              # chronological order in the prompt
    ctx = "\n\n".join(rs[i]["text"][:ROUND_CAP] for i in top)
    row = {"key": qid, "context": ctx, "n_rounds": len(rs), "embed_tokens": used,
           "hit_answer_session": any(rs[i]["sid"] in q["answer_session_ids"] for i in top)}
    cache_append("retrieval_vector", row)
    return row

vec = {r["key"]: r for r in pmap(vector_retrieve, sample, workers=8)}
print(f"Indexed {sum(r['n_rounds'] for r in vec.values()):,} rounds, "
      f"{sum(r['embed_tokens'] for r in vec.values()) / 1e6:.1f}M embedding tokens")
ans_q = [q for q in sample if not is_abs(by_id[q])]
print(f"Top-{K_VEC} contains a round from an evidence session: "
      f"{sum(vec[q]['hit_answer_session'] for q in ans_q)}/{len(ans_q)} answerable questions")

Indexed 26,798 rounds, 11.7M embedding tokens
Top-10 contains a round from an evidence session: 94/96 answerable questions


## (b) Bi-temporal graph memory — `add_episode` for every session (slides 53-54)

Each session is one **episode**. `gpt-4o-mini` extracts typed facts with Pydantic structured output: subject,
relation, object, a self-contained fact sentence, the turn it came from, an optional event date, and whether the fact
is a **state** (a value that replaces earlier values: where the user lives, how many X they own, their current
goal). Episodes are ingested in chronological order:

1. **Entity resolution** — normalised names first (`I` / `me` / `the user` → `user`), then embedding similarity
   ≥ 0.92 between entity names of the same history (cheap and certain first, as on slide 48).
2. **Invalidate, don't delete** — a *state* fact from a later episode with the same subject and relation but a
   different object sets `invalid_at` on the older edge; the old edge stays queryable.
3. **Provenance** — every edge keeps `session_id`, `turn` and `valid_at` (event time) / `created_at` (ingestion time).

Retrieval embeds the facts, ranks them against the question (with a small boost for entities named in the question)
and returns the 25 best facts *with their validity interval*, plus the source turns of the 5 best facts — the source
text stays attached (slide 38).

In [4]:
class Fact(BaseModel):
    subject: str = Field(description="entity the fact is about; use 'user' for the human user")
    relation: str = Field(description="UPPER_SNAKE_CASE relation, e.g. LIVES_IN, OWNS, PLANS_TO, RECOMMENDED")
    object: str = Field(description="value or entity; keep numbers, names and dates exact")
    fact: str = Field(description="one self-contained sentence with all specifics (names, numbers, dates)")
    turn: int = Field(description="index of the turn the fact comes from")
    event_date: Optional[str] = Field(description="YYYY/MM/DD if the fact has its own date (resolve 'last week' against the session date), else null")
    is_state: bool = Field(description="true if this value replaces earlier values of the same subject+relation (current job, address, count, status); false for events and multi-valued facts")

class Episode(BaseModel):
    facts: list[Fact]

EXTRACT_SYS = ("You build the long-term memory of a personal assistant. From one chat session, extract the facts "
               "worth remembering: everything the USER says about themselves (events, possessions, preferences, "
               "plans, people, places, numbers, dates, changes of state) and specific things the ASSISTANT "
               "provided (recommendations, names, numbers, lists the user may ask about later). Skip generic "
               "chit-chat. Be exhaustive for personal details; at most 40 facts.")
TURN_CAP, SESSION_CAP = 4000, 40000

def session_key(sid, sess):
    return hashlib.sha1((sid + json.dumps(sess)).encode()).hexdigest()

ext_cache = cache_load("extractions")
def extract(job):
    sid, date, sess = job
    key = session_key(sid, sess)
    if key in ext_cache:
        return ext_cache[key]
    need_live("fact extraction")
    turns = "\n".join(f"[turn {i}] {t['role'].upper()}: {t['content'][:TURN_CAP]}" for i, t in enumerate(sess))
    msgs = [{"role": "system", "content": EXTRACT_SYS},
            {"role": "user", "content": f"Session date: {date}\n\n{turns[:SESSION_CAP]}"}]
    try:
        try:
            r = client.chat.completions.parse(model=CHAT_MODEL, temperature=0, seed=7, max_tokens=6000,
                                              response_format=Episode, messages=msgs)
        except Exception:                            # rare degenerate output at T=0: one retry at T=0.3
            r = with_retry(client.chat.completions.parse, model=CHAT_MODEL, temperature=0.3, seed=7,
                           max_tokens=6000, response_format=Episode, messages=msgs)
        facts = [f.model_dump() for f in (r.choices[0].message.parsed.facts if r.choices[0].message.parsed else [])]
        row = {"key": key, "sid": sid, "facts": facts,
               "in_tok": r.usage.prompt_tokens, "out_tok": r.usage.completion_tokens}
    except Exception as e:                           # e.g. output too long to parse: keep the episode, no facts
        row = {"key": key, "sid": sid, "facts": [], "error": type(e).__name__, "in_tok": 0, "out_tok": 0}
    cache_append("extractions", row)
    ext_cache[key] = row
    return row

jobs = list({session_key(s, x): (s, d, x) for q in sample for s, d, x in
             zip(by_id[q]["haystack_session_ids"], by_id[q]["haystack_dates"], by_id[q]["haystack_sessions"])}.values())
t0 = time.time()
ext_rows = pmap(extract, jobs, workers=48)
EXT = {r["key"]: r for r in ext_rows}
print(f"Episodes extracted: {len(ext_rows):,} sessions "
      f"({sum('error' in r for r in ext_rows)} failed, kept without facts) | "
      f"{sum(len(r['facts']) for r in ext_rows):,} facts | LLM tokens in {sum(r['in_tok'] for r in ext_rows) / 1e6:.1f}M, "
      f"out {sum(r['out_tok'] for r in ext_rows) / 1e6:.2f}M")

Episodes extracted: 4,848 sessions (0 failed, kept without facts) | 87,628 facts | LLM tokens in 12.8M, out 4.67M


In [5]:
SELF = {"i", "me", "my", "myself", "user", "the user", "you (user)", "human"}
def norm(name):
    n = re.sub(r"[^a-z0-9 ]+", " ", str(name).lower()).strip()
    n = re.sub(r"^(the|a|an|my) ", "", n)
    return "user" if n in SELF or str(name).strip().lower() in SELF else n

def build_graph(qid):
    # Ingest episodes chronologically; returns edges with validity intervals + entity-resolution stats.
    q = by_id[qid]
    order = sorted(range(len(q["haystack_dates"])), key=lambda i: q["haystack_dates"][i])
    edges = []
    for i in order:
        sid, date, sess = q["haystack_session_ids"][i], q["haystack_dates"][i], q["haystack_sessions"][i]
        for f in EXT[session_key(sid, sess)]["facts"]:
            t = f["turn"] if isinstance(f["turn"], int) and 0 <= f["turn"] < len(sess) else 0
            edges.append({"s": norm(f["subject"]), "r": f["relation"].upper(), "o": f["object"], "fact": f["fact"],
                          "sid": sid, "turn": t, "created_at": date[:10],
                          "valid_at": (f.get("event_date") or date[:10]), "invalid_at": None,
                          "state": bool(f["is_state"]), "src": sess[t]["content"]})
    return edges

def resolve_and_invalidate(edges, emb):
    # Stage 2 of entity resolution: merge entity names whose embeddings are >= 0.92 similar.
    names = sorted({e["s"] for e in edges} | {norm(e["o"]) for e in edges})
    canon = {n: n for n in names}
    if len(names) > 1:
        V = emb(names)
        S = V @ V.T
        for a in range(len(names)):
            for b in np.where(S[a, a + 1:] >= 0.92)[0] + a + 1:
                canon[names[b]] = canon[names[a]]
    merged = sum(1 for n in names if canon[n] != n)
    for e in edges:
        e["s"] = canon[e["s"]]
    # Invalidate, don't delete: a newer state fact closes the older edge with the same subject + relation.
    latest, invalidated = {}, 0
    for e in edges:                                   # edges are already in ingestion order
        if not e["state"]:
            continue
        k = (e["s"], e["r"])
        old = latest.get(k)
        if (old is not None and old["sid"] != e["sid"]          # only a later episode can supersede
                and canon.get(norm(old["o"]), norm(old["o"])) != canon.get(norm(e["o"]), norm(e["o"]))):
            old["invalid_at"] = e["valid_at"]
            invalidated += 1
        latest[k] = e
    return merged, invalidated

K_FACTS, K_SRC, SRC_CAP = 25, 5, 1200
gr_cache = cache_load("retrieval_graph")
def graph_retrieve(qid):
    if qid in gr_cache:
        return gr_cache[qid]
    need_live("graph retrieval")
    q = by_id[qid]
    edges = build_graph(qid)
    used = [0]
    def emb(texts):
        V, u = embed(texts, cap=2000); used[0] += u; return V
    merged, invalidated = resolve_and_invalidate(edges, emb)
    V = emb([f"{e['s']} {e['r']} {e['o']}: {e['fact']}" for e in edges] + [q["question"]])
    sims = V[:-1] @ V[-1]
    qn = norm(q["question"])
    sims += np.array([0.05 if len(e["s"]) > 3 and e["s"] != "user" and e["s"] in qn else 0.0 for e in edges])
    top = list(np.argsort(-sims)[:K_FACTS])
    lines = []
    for i in sorted(top, key=lambda i: edges[i]["valid_at"]):
        e = edges[i]
        status = f"superseded {e['invalid_at']}" if e["invalid_at"] else "current"
        lines.append(f"- [{e['valid_at']} | {status}] {e['fact']}  (session {e['sid']}, turn {e['turn']})")
    srcs = [f"[{edges[i]['created_at']} session {edges[i]['sid']} turn {edges[i]['turn']}] {edges[i]['src'][:SRC_CAP]}"
            for i in top[:K_SRC]]
    ctx = "FACTS FROM MEMORY GRAPH:\n" + "\n".join(lines) + "\n\nSOURCE TURNS OF THE TOP FACTS:\n" + "\n\n".join(srcs)
    row = {"key": qid, "context": ctx, "n_edges": len(edges), "n_entities": len({e['s'] for e in edges}),
           "merged_entities": merged, "invalidated": invalidated, "embed_tokens": used[0],
           "hit_answer_session": any(edges[i]["sid"] in q["answer_session_ids"] for i in top)}
    cache_append("retrieval_graph", row)
    return row

gr = {r["key"]: r for r in pmap(graph_retrieve, sample, workers=8)}
print(f"Graph memories: {sum(r['n_edges'] for r in gr.values()):,} edges in {len(gr)} histories "
      f"(avg {np.mean([r['n_edges'] for r in gr.values()]):.0f} edges, "
      f"{np.mean([r['merged_entities'] for r in gr.values()]):.0f} entity merges by embedding)")
print(f"Edges invalidated (kept, with invalid_at): {sum(r['invalidated'] for r in gr.values()):,}")
print(f"Top-{K_FACTS} facts come from an evidence session: "
      f"{sum(gr[q]['hit_answer_session'] for q in ans_q)}/{len(ans_q)} answerable questions")

Graph memories: 92,002 edges in 108 histories (avg 852 edges, 10 entity merges by embedding)
Edges invalidated (kept, with invalid_at): 4,442
Top-25 facts come from an evidence session: 96/96 answerable questions


### What an invalidated edge looks like

A knowledge-update question from the sample, shown as the answerer sees it: older values are not deleted, they carry
the date on which a newer episode superseded them (slide 53).

Which facts count as *state* is decided by the extractor, and the label is noisy: some opinions or plans get
superseded too. Nothing is lost when that happens — the edge stays in the graph with its dates and source — but it is
the same lesson as slide 50: a structurally valid write is not automatically a correct one.

In [6]:
ku = next(q for q in sample if by_id[q]["question_type"] == "knowledge-update" and "superseded" in gr[q]["context"])
print(f"Q ({by_id[ku]['question_date']}): {by_id[ku]['question']}\nGold: {by_id[ku]['answer']}\n")
facts = gr[ku]["context"].split("\n\nSOURCE TURNS")[0].splitlines()[1:]
for line in [l for l in facts if "superseded" in l][:3] + [l for l in facts if "current" in l][:4]:
    print(line[:220])

Q (2023/10/26 (Thu) 11:44): How much did I earn at the Downtown Farmers Market on my most recent visit?
Gold: $420

- [2023/04/11 | superseded 2023/06/21] User believes that people who visit the Downtown Farmers Market have a strong emotional connection to the event and the community.  (session answer_c9f5693c_1, turn 11)
- [2023/04/11 | superseded 2023/06/21] User wants to cross-promote the Downtown Farmers Market through the themed gift basket.  (session answer_c9f5693c_1, turn 11)
- [2023/04/11 | superseded 2023/09/30] User sells artisanal soaps.  (session answer_c9f5693c_1, turn 7)
- [2023/03/18 | current] User participated in the Downtown Farmers Market on March 18th and May 6th.  (session answer_c9f5693c_1, turn 2)
- [2023/03/25 | current] User participated in the Arts and Crafts Fair on March 25th.  (session answer_c9f5693c_1, turn 2)
- [2023/04/01 | current] User participated in the Earth Day Market on April 1st.  (session answer_c9f5693c_1, turn 2)
- [2023/04/11 | current] Use

## Answer and judge — same answerer, same official judge prompts

The answerer gets the question date and the retrieved memory, reasons briefly and answers (or says the information
is not available). The judge prompts below are LongMemEval's own, one per question type plus one for abstention.
**(c) Full context** is run on a fixed subset of 28 sampled questions (4 per type + 4 abstention): each costs
≈120k input tokens, so the full sample would cost as much as the whole graph index.

In [7]:
def anscheck_prompt(task, question, answer, response, abstention=False):
    # Copied from LongMemEval src/evaluation/evaluate_qa.py (get_anscheck_prompt).
    if not abstention:
        if task in ['single-session-user', 'single-session-assistant', 'multi-session']:
            template = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response is equivalent to the correct answer or contains all the intermediate steps to get the correct answer, you should also answer yes. If the response only contains a subset of the information required by the answer, answer no. \n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        elif task == 'temporal-reasoning':
            template = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response is equivalent to the correct answer or contains all the intermediate steps to get the correct answer, you should also answer yes. If the response only contains a subset of the information required by the answer, answer no. In addition, do not penalize off-by-one errors for the number of days. If the question asks for the number of days/weeks/months, etc., and the model makes off-by-one errors (e.g., predicting 19 days when the answer is 18), the model's response is still correct. \n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        elif task == 'knowledge-update':
            template = "I will give you a question, a correct answer, and a response from a model. Please answer yes if the response contains the correct answer. Otherwise, answer no. If the response contains some previous information along with an updated answer, the response should be considered as correct as long as the updated answer is the required answer.\n\nQuestion: {}\n\nCorrect Answer: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        elif task == 'single-session-preference':
            template = "I will give you a question, a rubric for desired personalized response, and a response from a model. Please answer yes if the response satisfies the desired response. Otherwise, answer no. The model does not need to reflect all the points in the rubric. The response is correct as long as it recalls and utilizes the user's personal information correctly.\n\nQuestion: {}\n\nRubric: {}\n\nModel Response: {}\n\nIs the model response correct? Answer yes or no only."
        else:
            raise NotImplementedError
    else:
        template = "I will give you an unanswerable question, an explanation, and a response from a model. Please answer yes if the model correctly identifies the question as unanswerable. The model could say that the information is incomplete, or some other information is given but the asked information is not.\n\nQuestion: {}\n\nExplanation: {}\n\nModel Response: {}\n\nDoes the model correctly identify the question as unanswerable? Answer yes or no only."
    return template.format(question, answer, response)

ANSWER_SYS = ("You are a personal assistant with a memory of your past conversations with the user. Use only the "
              "memory provided. Dates matter: compute elapsed time from the dates shown, and prefer the most recent "
              "value when a fact changed. If the memory does not contain the answer, say so. Think briefly, then "
              "give the final answer on the last line as 'Answer: ...'.")

FULL_SUBSET = [q for t in TYPES for q in [x for x in sample if group(x) == t][:4]] + \
              [q for q in sample if group(q) == "abstention"][:4]

def full_context(qid, max_tokens=118_000):
    q = by_id[qid]
    sess = [f"### Session {sid} ({d})\n" + "\n".join(f"{t['role'].upper()}: {t['content']}" for t in s)
            for sid, d, s in sorted(zip(q["haystack_session_ids"], q["haystack_dates"], q["haystack_sessions"]),
                                    key=lambda x: x[1])]
    while sum(map(len, sess)) / 3.6 > max_tokens:      # drop the oldest sessions until it fits
        sess.pop(0)
    return "\n\n".join(sess)

ans_cache, judge_cache = cache_load("answers"), cache_load("judgments")
def answer(job):
    qid, method = job
    key = f"{method}|{qid}"
    if key in ans_cache:
        return ans_cache[key]
    need_live(f"answer {key}")
    q = by_id[qid]
    ctx = {"vector": lambda: vec[qid]["context"], "graph": lambda: gr[qid]["context"],
           "full": lambda: full_context(qid)}[method]()
    r = with_retry(client.chat.completions.create, model=CHAT_MODEL, temperature=0, seed=7, messages=[
        {"role": "system", "content": ANSWER_SYS},
        {"role": "user", "content": f"Current date: {q['question_date']}\n\nMEMORY:\n{ctx}\n\nQuestion: {q['question']}"}])
    row = {"key": key, "answer": r.choices[0].message.content,
           "in_tok": r.usage.prompt_tokens, "out_tok": r.usage.completion_tokens}
    cache_append("answers", row)
    return row

def judge(job):
    qid, method = job
    key = f"{method}|{qid}"
    if key in judge_cache:
        return judge_cache[key]
    need_live(f"judgment {key}")
    q = by_id[qid]
    prompt = anscheck_prompt(q["question_type"], q["question"], q["answer"], ANS[key]["answer"], abstention=is_abs(q))
    r = with_retry(client.chat.completions.create, model=CHAT_MODEL, temperature=0, seed=7, max_tokens=5,
                   messages=[{"role": "user", "content": prompt}])
    row = {"key": key, "correct": "yes" in r.choices[0].message.content.lower(),
           "in_tok": r.usage.prompt_tokens, "out_tok": r.usage.completion_tokens}
    cache_append("judgments", row)
    return row

jobs = [(q, m) for m in ["vector", "graph"] for q in sample] + [(q, "full") for q in FULL_SUBSET]
ANS = {r["key"]: r for r in pmap(answer, jobs[:2 * len(sample)], workers=16)}
ANS.update({r["key"]: r for r in pmap(answer, jobs[2 * len(sample):], workers=3)})
JUD = {r["key"]: r for r in pmap(judge, jobs, workers=16)}
print(f"{len(ANS)} answers, {len(JUD)} judgments")

244 answers, 244 judgments


In [8]:
def acc(method, qids):
    k = sum(JUD[f"{method}|{q}"]["correct"] for q in qids)
    lo, hi = wilson(k, len(qids))
    return k, len(qids), f"{k / len(qids):5.0%} [{lo:.0%}-{hi:.0%}]"

GROUPS = TYPES + ["abstention"]
print(f"{'question group':<27}{'n':>3} | {'(a) vector memory':^19} | {'(b) bi-temporal graph':^21}")
print("-" * 76)
for g in GROUPS:
    qs = [q for q in sample if group(q) == g]
    print(f"{g:<27}{len(qs):>3} | {acc('vector', qs)[2]:^19} | {acc('graph', qs)[2]:^21}")
print("-" * 76)
print(f"{'overall':<27}{len(sample):>3} | {acc('vector', sample)[2]:^19} | {acc('graph', sample)[2]:^21}")

both = sum(JUD[f'vector|{q}']['correct'] and JUD[f'graph|{q}']['correct'] for q in sample)
v_only = sum(JUD[f'vector|{q}']['correct'] and not JUD[f'graph|{q}']['correct'] for q in sample)
g_only = sum(JUD[f'graph|{q}']['correct'] and not JUD[f'vector|{q}']['correct'] for q in sample)
print(f"\nPer question: both correct {both}, only vector {v_only}, only graph {g_only} "
      f"-> an oracle choosing per question would reach {(both + v_only + g_only) / len(sample):.0%}")

print(f"\nFull-context subset ({len(FULL_SUBSET)} questions, 4 per group):")
for m in ["full", "vector", "graph"]:
    print(f"  {m:<7} {acc(m, FULL_SUBSET)[2]}   avg answer-prompt tokens "
          f"{np.mean([ANS[f'{m}|{q}']['in_tok'] for q in FULL_SUBSET]):>9,.0f}")

question group               n |  (a) vector memory  | (b) bi-temporal graph
----------------------------------------------------------------------------
single-session-user         16 |     94% [72%-99%]   |      94% [72%-99%]   
single-session-assistant    16 |     94% [72%-99%]   |      88% [64%-97%]   
single-session-preference   16 |     19% [7%-43%]    |      50% [28%-72%]   
multi-session               16 |     69% [44%-86%]   |      88% [64%-97%]   
knowledge-update            16 |   100% [81%-100%]   |    100% [81%-100%]   
temporal-reasoning          16 |     69% [44%-86%]   |      69% [44%-86%]   
abstention                  12 |   100% [76%-100%]   |    100% [76%-100%]   
----------------------------------------------------------------------------
overall                    108 |     77% [68%-84%]   |      83% [75%-89%]   

Per question: both correct 78, only vector 5, only graph 12 -> an oracle choosing per question would reach 88%

Full-context subset (28 questions, 4 per

In [9]:
ctx_tok = {m: np.mean([ANS[f"{m}|{q}"]["in_tok"] for q in sample]) for m in ["vector", "graph"]}
ext_in, ext_out = sum(r["in_tok"] for r in EXT.values()), sum(r["out_tok"] for r in EXT.values())
emb_vec = sum(r["embed_tokens"] for r in vec.values())
emb_gr = sum(r["embed_tokens"] for r in gr.values())
llm_in = ext_in + sum(r["in_tok"] for r in ANS.values()) + sum(r["in_tok"] for r in JUD.values())
llm_out = ext_out + sum(r["out_tok"] for r in ANS.values()) + sum(r["out_tok"] for r in JUD.values())
ov = {m: acc(m, sample) for m in ["vector", "graph"]}
tr = {m: acc(m, [q for q in sample if group(q) == "temporal-reasoning"]) for m in ["vector", "graph"]}
sa = {m: acc(m, [q for q in sample if group(q) == "single-session-assistant"]) for m in ["vector", "graph"]}
ku_ = {m: acc(m, [q for q in sample if group(q) == "knowledge-update"]) for m in ["vector", "graph"]}
fc = {m: acc(m, FULL_SUBSET) for m in ["full", "vector", "graph"]}

print("=" * 86)
print(f"RESULTS SUMMARY — Notebook 10: agent memory on LongMemEval-S (cleaned), {len(sample)} stratified questions")
print("=" * 86)
print(f"Accuracy, official judge prompts (gpt-4o-mini judge)  vector {ov['vector'][2]}  |  graph {ov['graph'][2]}")
print(f"  temporal-reasoning                                  vector {tr['vector'][2]}  |  graph {tr['graph'][2]}")
print(f"  knowledge-update                                    vector {ku_['vector'][2]}  |  graph {ku_['graph'][2]}")
print(f"  single-session-assistant                            vector {sa['vector'][2]}  |  graph {sa['graph'][2]}")
print(f"Full-context subset (n={len(FULL_SUBSET)}): full {fc['full'][2]} | vector {fc['vector'][2]} | graph {fc['graph'][2]}")
print(f"Answer prompt tokens / question: vector {ctx_tok['vector']:,.0f} | graph {ctx_tok['graph']:,.0f} | "
      f"full context {np.mean([ANS[f'full|{q}']['in_tok'] for q in FULL_SUBSET]):,.0f}")
print(f"Graph index: {len(EXT):,} episodes -> {sum(len(r['facts']) for r in EXT.values()):,} facts, "
      f"{ext_in / 1e6:.1f}M input + {ext_out / 1e6:.2f}M output LLM tokens; "
      f"{sum(r['invalidated'] for r in gr.values()):,} edges invalidated, none deleted")
print(f"Vector index: {emb_vec / 1e6:.1f}M embedding tokens, 0 LLM calls  (graph embeddings {emb_gr / 1e6:.1f}M)")
print(f"Total LLM tokens for this run: {llm_in / 1e6:.1f}M in / {llm_out / 1e6:.2f}M out "
      f"(+ {(emb_vec + emb_gr) / 1e6:.1f}M embedding tokens)")
print("\nLecture message (Slides 53-56): a bi-temporal graph memory pays an LLM extraction pass over every episode;")
print("compare the per-type rows above with slide 55 (Zep: +38.4% temporal, but single-session-assistant 94.6% -> 80.4%)")
print("and slide 56 — no memory design wins every question type, so measure by type before choosing one.")

RESULTS SUMMARY — Notebook 10: agent memory on LongMemEval-S (cleaned), 108 stratified questions
Accuracy, official judge prompts (gpt-4o-mini judge)  vector   77% [68%-84%]  |  graph   83% [75%-89%]
  temporal-reasoning                                  vector   69% [44%-86%]  |  graph   69% [44%-86%]
  knowledge-update                                    vector  100% [81%-100%]  |  graph  100% [81%-100%]
  single-session-assistant                            vector   94% [72%-99%]  |  graph   88% [64%-97%]
Full-context subset (n=28): full   75% [57%-87%] | vector   79% [60%-90%] | graph   89% [73%-96%]
Answer prompt tokens / question: vector 3,170 | graph 1,823 | full context 88,091
Graph index: 4,848 episodes -> 87,628 facts, 12.8M input + 4.67M output LLM tokens; 4,442 edges invalidated, none deleted
Vector index: 11.7M embedding tokens, 0 LLM calls  (graph embeddings 3.1M)
Total LLM tokens for this run: 15.8M in / 4.68M out (+ 14.8M embedding tokens)

Lecture message (Slides 53-56): 